In [1]:
import os

In [2]:
%pwd

'/Users/admin/PycharmProjects/AnimalsDetectionMLOPS/notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/admin/PycharmProjects/AnimalsDetectionMLOPS'

In [5]:
from dataclasses import dataclass
from pathlib import Path
from typing import List

@dataclass(frozen=True)
class EvaluationConfig:
    root_dir: Path
    trained_model_path: Path
    data_root: Path
    params_image_size: List[int]

In [6]:
from src.cnnClassifier.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.cnnClassifier.utils.common import read_yaml, create_directories
import torch
from torchvision import models

class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        cfg = self.config
        return EvaluationConfig(
            root_dir=Path(cfg.training.root_dir),
            trained_model_path=Path(cfg.training.trained_model_path),
            data_root=Path(cfg.data_ingestion.unzip_dir),
            params_image_size=self.params.IMAGE_SIZE,
        )

In [7]:
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from PIL import Image
import json

class Evaluator:
    def __init__(self, config: EvaluationConfig):
        self.config = config
        self.device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
        self.model = self._load_model().to(self.device)
        self.transform = transforms.Compose([
            transforms.Resize((self.config.params_image_size[0], self.config.params_image_size[1])),
            transforms.ToTensor()
        ])
        self.criterion = nn.CrossEntropyLoss()

    def _load_model(self):
        # build VGG16 with 2-class head, then load state_dict
        model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        in_features = model.classifier[6].in_features
        # infer classes from folders
        try:
            class_dirs = [d for d in os.listdir(self.config.data_root) if (self.config.data_root / d).is_dir()]
            num_classes = len(class_dirs) if len(class_dirs) > 0 else 2
        except Exception:
            num_classes = 2
        model.classifier[6] = nn.Linear(in_features, num_classes)
        # load trained weights if present
        path = self.config.trained_model_path
        if path.exists():
            try:
                sd = torch.load(path, weights_only=True, map_location=self.device)
                if hasattr(sd, 'state_dict'):
                    sd = sd.state_dict()
                model.load_state_dict(sd, strict=False)
            except Exception:
                try:
                    sd = torch.load(path, weights_only=False, map_location=self.device)
                    if hasattr(sd, 'state_dict'):
                        sd = sd.state_dict()
                    model.load_state_dict(sd, strict=False)
                except Exception:
                    pass
        model.eval()
        return model

    def build_dataset(self):
        dataset = datasets.ImageFolder(root=str(self.config.data_root), transform=self.transform)
        loader = DataLoader(dataset, batch_size=32, shuffle=False)
        return dataset, loader

    def evaluate(self):
        dataset, loader = self.build_dataset()
        y_true = []
        y_pred = []
        total_loss = 0.0
        total_items = 0
        with torch.no_grad():
            for inputs, labels in loader:
                inputs = inputs.to(self.device)
                labels = labels.to(self.device)
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                total_loss += float(loss.item()) * inputs.size(0)
                total_items += inputs.size(0)
                preds = outputs.argmax(dim=1).cpu().numpy()
                y_pred.extend(preds.tolist())
                y_true.extend(labels.cpu().numpy().tolist())
        avg_loss = total_loss / max(total_items, 1)
        # metrics
        target_names = dataset.classes
        print(f"Classes: {target_names}")
        print("Classification Report:")
        print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))
        print("Confusion Matrix:")
        print(confusion_matrix(y_true, y_pred))
        # accuracy
        acc = np.mean(np.array(y_true) == np.array(y_pred))
        # per-class accuracy
        y_true_arr = np.array(y_true)
        y_pred_arr = np.array(y_pred)
        per_class_acc = {}
        for i, name in enumerate(target_names):
            mask = y_true_arr == i
            per_class_acc[name] = float((y_pred_arr[mask] == i).mean()) if mask.any() else 0.0
        print(f"Loss: {avg_loss:.6f}")
        print(f"Accuracy: {acc:.4f}")
        print({"per_class_accuracy": per_class_acc})
        # write scores.json at project root
        scores_path = Path.cwd() / "scores.json"
        with open(scores_path, "w") as f:
            json.dump({"loss": avg_loss, "accuracy": float(acc)}, f)
        print(f"Saved scores to {scores_path}")
        return avg_loss, acc

    def predict_image(self, image_path: str):
        img = Image.open(image_path).convert('RGB')
        x = self.transform(img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits = self.model(x)
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            idx = int(np.argmax(probs))
        # map index to class names
        class_names = sorted([d for d in os.listdir(self.config.data_root) if (self.config.data_root / d).is_dir()])
        label = class_names[idx] if class_names else ('dogs_set' if idx == 1 else 'cats_set')
        print({"label": label, "probs": {class_names[i]: float(probs[i]) for i in range(len(class_names))}})
        return label

In [8]:
# Run evaluation
try:
    cfg_mgr = ConfigurationManager()
    eval_cfg = cfg_mgr.get_evaluation_config()
    evaluator = Evaluator(config=eval_cfg)
    evaluator.evaluate()
    # Example single prediction (adjust path as needed)
    sample_img = str(eval_cfg.data_root / 'cats_set' / 'cat.4001.jpg')
    if os.path.exists(sample_img):
        print("Single image prediction:")
        evaluator.predict_image(sample_img)
except Exception as e:
    print(f"Error during evaluation: {e}")

[2026-01-22 14:03:47,382: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-01-22 14:03:47,383: INFO: common: yaml file: params.yaml loaded successfully]
[2026-01-22 14:03:47,384: INFO: common: created directory at: artifacts]
Classes: ['cats_set', 'dogs_set']
Classification Report:
              precision    recall  f1-score   support

    cats_set       0.50      1.00      0.67       500
    dogs_set       0.00      0.00      0.00       500

    accuracy                           0.50      1000
   macro avg       0.25      0.50      0.33      1000
weighted avg       0.25      0.50      0.33      1000

Confusion Matrix:
[[500   0]
 [500   0]]
Loss: 0.744469
Accuracy: 0.5000
{'per_class_accuracy': {'cats_set': 1.0, 'dogs_set': 0.0}}
Saved scores to /Users/admin/PycharmProjects/AnimalsDetectionMLOPS/scores.json
Single image prediction:
{'label': 'cats_set', 'probs': {'cats_set': 0.6483012437820435, 'dogs_set': 0.35169875621795654}}
